# Análise interativa - Solução 3 OpenCV assistido

**Autor:** Manoel Furtado  
**Objetivo:** entender e ajustar a segmentação de fissuras por OpenCV implementada em `solucao_3_opencv_assistido.py`.

Este notebook mostra as etapas intermediárias do pipeline:

1. carregar imagem e label de segmentação;
2. converter o label YOLO em máscara de referência;
3. aplicar CLAHE, black-hat, Otsu, Canny e morfologia;
4. comparar máscara prevista com máscara real;
5. testar combinações de `kernel`, `canny_low`, `canny_high` e `min_component_area`;
6. executar o script completo opcionalmente.

## 1. Importações e caminhos

A célula localiza a pasta da solução e importa as funções do script original.

In [ ]:
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

SCRIPT_NAME = 'solucao_3_opencv_assistido.py'

def find_solution_dir(script_name):
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / script_name).exists():
            return candidate
    matches = list(start.rglob(script_name))
    if not matches:
        raise FileNotFoundError(f'Não encontrei {script_name} a partir de {start}')
    return matches[0].parent

SOLUTION_DIR = find_solution_dir(SCRIPT_NAME)
DESAFIO_DIR = SOLUTION_DIR.parents[1]
DATA_DIR = DESAFIO_DIR / 'data'
OUTPUT_DIR = SOLUTION_DIR / 'resultados' / 'opencv_assistido'

sys.path.insert(0, str(SOLUTION_DIR))

from solucao_3_opencv_assistido import crack_mask, label_to_mask, metrics, overlay, parse_args, process, remove_small_components

print('Solução:', SOLUTION_DIR)
print('Dados   :', DATA_DIR)
print('Saída   :', OUTPUT_DIR)

## 2. Escolha de imagem

Troque `image_index` para testar outras imagens. Use `limit` mais adiante para avaliar várias imagens rapidamente.

In [ ]:
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
image_paths = [path for path in sorted((DATA_DIR / 'images').iterdir()) if path.suffix.lower() in IMAGE_SUFFIXES]
print('Imagens encontradas:', len(image_paths))

for index, path in enumerate(image_paths[:10]):
    print(index, path.name)

image_index = 0
image_path = image_paths[image_index]
label_path = DATA_DIR / 'labels' / f'{image_path.stem}.txt'

image_bgr = cv2.imread(str(image_path))
height, width = image_bgr.shape[:2]
true_mask = label_to_mask(cv2, np, label_path, width, height)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title(image_path.name)
axes[0].axis('off')
axes[1].imshow(true_mask, cmap='gray')
axes[1].set_title('Máscara real')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 3. Parâmetros principais

Altere os valores abaixo e execute as próximas células para ver como cada parâmetro muda a máscara.

In [ ]:
kernel = 17
canny_low = 40
canny_high = 140
min_component_area = 25

args = parse_args([
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--kernel', str(kernel),
    '--canny-low', str(canny_low),
    '--canny-high', str(canny_high),
    '--min-component-area', str(min_component_area),
])

print(args)

## 4. Visualização das etapas internas

Esta célula replica as etapas de `crack_mask` para deixar visível o que cada filtro faz.

In [ ]:
gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)

kernel_line = cv2.getStructuringElement(cv2.MORPH_RECT, (args.kernel, args.kernel))
blackhat = cv2.morphologyEx(clahe, cv2.MORPH_BLACKHAT, kernel_line)
blackhat = cv2.normalize(blackhat, None, 0, 255, cv2.NORM_MINMAX)

_, threshold = cv2.threshold(blackhat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
edges = cv2.Canny(clahe, args.canny_low, args.canny_high)
combined = cv2.bitwise_or(threshold, edges)

kernel_small = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
closed = cv2.morphologyEx(combined, cv2.MORPH_CLOSE, kernel_small, iterations=1)
pred_mask = remove_small_components(cv2, np, closed, args.min_component_area)

stages = [
    ('Cinza', gray),
    ('CLAHE', clahe),
    ('Black-hat', blackhat),
    ('Otsu', threshold),
    ('Canny', edges),
    ('Final', pred_mask),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for axis, (title, stage) in zip(axes.ravel(), stages):
    axis.imshow(stage, cmap='gray')
    axis.set_title(title)
    axis.axis('off')
plt.tight_layout()
plt.show()

## 5. Métricas e sobreposição

Agora comparamos a máscara prevista pelo OpenCV com a máscara real do label.

In [ ]:
pred_mask = crack_mask(cv2, np, image_bgr, args)
scores = metrics(np, pred_mask, true_mask)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(true_mask, cmap='gray')
axes[0].set_title('Real')
axes[0].axis('off')
axes[1].imshow(pred_mask, cmap='gray')
axes[1].set_title('Predita')
axes[1].axis('off')
axes[2].imshow(cv2.cvtColor(overlay(cv2, np, image_bgr, pred_mask), cv2.COLOR_BGR2RGB))
axes[2].set_title('Predição sobreposta')
axes[2].axis('off')
plt.tight_layout()
plt.show()

print(scores)

## 6. Busca simples de parâmetros em uma imagem

Comece com poucos valores. Depois aumente as listas se quiser refinar. A ordenação abaixo prioriza maior Dice e maior IoU.

In [ ]:
from itertools import product

search_space = {
    'kernel': [9, 13, 17, 21],
    'canny_low': [30, 40, 60],
    'canny_high': [100, 140, 180],
    'min_component_area': [10, 25, 50],
}

results = []
keys = list(search_space.keys())
for values in product(*(search_space[key] for key in keys)):
    params = dict(zip(keys, values))
    test_args = parse_args([
        '--data-dir', str(DATA_DIR),
        '--output-dir', str(OUTPUT_DIR),
        '--kernel', str(params['kernel']),
        '--canny-low', str(params['canny_low']),
        '--canny-high', str(params['canny_high']),
        '--min-component-area', str(params['min_component_area']),
    ])
    pred = crack_mask(cv2, np, image_bgr, test_args)
    row = metrics(np, pred, true_mask)
    row.update(params)
    results.append(row)

results = sorted(results, key=lambda item: (item['dice'], item['iou']), reverse=True)
for row in results[:10]:
    print(row)

## 7. Avaliação rápida em várias imagens

Use `sample_limit` para não processar o dataset inteiro durante os testes.

In [ ]:
sample_limit = 20
sample_rows = []

for path in image_paths[:sample_limit]:
    image = cv2.imread(str(path))
    if image is None:
        continue
    h, w = image.shape[:2]
    pred = crack_mask(cv2, np, image, args)
    true = label_to_mask(cv2, np, DATA_DIR / 'labels' / f'{path.stem}.txt', w, h)
    row = {'image': path.name}
    row.update(metrics(np, pred, true))
    sample_rows.append(row)

for row in sample_rows[:10]:
    print(row)

if sample_rows:
    avg = {key: sum(float(row[key]) for row in sample_rows) / len(sample_rows) for key in ['precision', 'recall', 'iou', 'dice']}
    print('\nMédias:', {key: round(value, 4) for key, value in avg.items()})

## 8. Execução do script completo

A célula abaixo chama `process`, que salva o CSV de métricas. Use `save_images=True` se quiser gerar máscaras e overlays no disco.

In [ ]:
execute_process = False
save_images = False
limit = 0

argv = [
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--kernel', str(kernel),
    '--canny-low', str(canny_low),
    '--canny-high', str(canny_high),
    '--min-component-area', str(min_component_area),
    '--limit', str(limit),
]
if save_images:
    argv.append('--save-images')

if execute_process:
    process(parse_args(argv))
else:
    print('Execução completa desativada. Mude execute_process = True para gerar o CSV.')

## 9. Dicas de interpretação

- `kernel` maior destaca fissuras mais largas, mas pode apagar detalhes finos.
- `canny_low` e `canny_high` baixos aumentam sensibilidade e também ruído.
- `min_component_area` remove pontos pequenos; se ficar alto demais, remove fissuras curtas.
- Precision baixa indica muitos falsos positivos; Recall baixo indica fissuras perdidas.
- Dice e IoU são mais úteis para comparar máscaras porque medem sobreposição.